# Data Inspection

This notebook examines the structure, completeness, and consistency of the raw Sofia temperature dataset retrieved from Meteostat.

## Imports and loading data

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_FILE = Path("../data/raw/sofia_daily_temperature.csv")

df = pd.read_csv(DATA_FILE)

## Basic Overview

In [ ]:
display(df.head())
print(df.info())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
df["time"] = pd.to_datetime(df["time"])

print(df["time"].min())
print(df["time"].max())
print(df["time"].is_unique)

## Check for missing rows

In [ ]:
expected_dates = (df["time"].max() - df["time"].min()).days + 1
actual_dates = df["time"].nunique()

print("Expected dates:", expected_dates)
print("Actual dates:", actual_dates)
print("Missing dates:", expected_dates - actual_dates)

### Plot available days per year

In [ ]:
df["year"] = df["time"].dt.year

days_by_year = df.groupby("year")["time"].nunique()

days_by_year.plot(
    kind="bar",
    figsize=(16, 6),
)

plt.xlabel("Year")
plt.ylabel("Number of days with data")
plt.title("Daily Data Coverage by Year")
plt.tight_layout()
plt.show()

Coverage is relatively limited during parts of the earlier record, while it becomes much more complete from around 2000 onward. Several complete gaps are also present, including 1968–1972 and 1996.

### Check available days per month and season

In [ ]:
df["month"] = df["time"].dt.month

days_by_month = df.groupby("month")["time"].nunique()

print(days_by_month)

In [ ]:
season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
}

df["season"] = df["month"].map(season_map)

days_by_season = df.groupby("season")["time"].nunique()

print(days_by_season)

The overall number of available observations is relatively similar across seasons.

## Missing columns

In [ ]:
df.isna().sum()

Mean temperature (temp) is almost complete, while maximum temperature (tmax) has a relatively small amount of missing data. Minimum temperature (tmin) has substantially more missing observations.

### tmin inspection

In [ ]:
tmin_missing_percentage_by_year = (
    df.groupby("year")["tmin"]
    .apply(lambda x: x.isna().sum() / len(x) * 100)
    .round(2)
)

print(tmin_missing_percentage_by_year.to_string())

In [ ]:
tmin_missing_percentage_by_year.plot(
    figsize=(16, 6)
)

plt.xlabel("Year")
plt.ylabel("Missing Tmin (%)")
plt.title("Missing Minimum Temperature by Year")
plt.tight_layout()
plt.show()

The missing tmin values are concentrated in particular historical periods rather than being evenly distributed throughout the dataset. This will need to be considered when deciding how minimum-temperature data should be handled during cleaning.

## Check for logical consistency

In [ ]:
df[["temp", "tmin", "tmax"]].describe()

In [ ]:
display(df.loc[df["temp"].nsmallest(10).index, ["time", "temp", "tmin", "tmax"]])
display(df.loc[df["temp"].nlargest(10).index, ["time", "temp", "tmin", "tmax"]])

In [ ]:
print("Days where Tmin > Tmax:", (df["tmin"] > df["tmax"]).sum())

print("Days where Tavg is outside Tmin/Tmax:",
      ((df["temp"] < df["tmin"]) | (df["temp"] > df["tmax"])).sum())

The results show no major inconsistencies in the dataset.

## Save processed data

In [ ]:
processed_df = df.copy()

processed_df = processed_df.sort_values("time").reset_index(drop=True)

source_columns = [
    column for column in processed_df.columns
    if column.endswith("_source")
]

processed_df = processed_df.drop(columns=source_columns)

OUTPUT_DIR = Path("../data/processed")
OUTPUT_FILE = OUTPUT_DIR / "sofia_daily_temperature.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

processed_df.to_csv(OUTPUT_FILE, index=False)

print(f"Processed data saved")

## Conclusion

The main data-quality considerations are the uneven availability of observations across the historical period, with earlier years generally having more missing dates, and the substantial number of missing `tmin` values, particularly during the 1990s.

No duplicate dates or major logical inconsistencies were identified, so no rows need to be removed for those reasons.

Based on these findings, only minimal processing is necessary. The processed dataset will retain the available temperature observations and missing values, remove the source columns  and add `year`, `month`, and `season` columns for convenience during analysis.
